In [1]:
import sys

scopepath = '/Users/sergivela/Documents/SCOPE/Program/'
sys.path.append(scopepath)

import Scope
from Scope import *

In [6]:
unique_ligands_path = "/Users/sergivela/Documents/SCOPE/Database_SCO/7-Ligands/unique_ligands.npy"
ligands_with_opt_path = "/Users/sergivela/Documents/SCOPE/Database_SCO/7-Ligands/ligand_systems.npy"
#new_binary_folder = "/Users/sergivela/Documents/SCOPE/Database_SCO/7-Ligands/Unique_Ligands_Binaries/"
#new_calcs_folder  = "/Users/sergivela/Documents/SCOPE/Database_SCO/7-Ligands/Unique_Ligands_Binaries/1-Optimizations/"

if os.path.isfile(unique_ligands_path):
    unique_ligands = np.load(unique_ligands_path, allow_pickle=True)
    print(len(unique_ligands),"unique ligands found")

if os.path.isfile(ligands_with_opt_path):
    optimized_ligs = np.load(ligands_with_opt_path, allow_pickle=True)
    print(len(optimized_ligs),"optimized ligands found")

556 unique ligands found
555 optimized ligands found


In [7]:
for idx, olig in enumerate(optimized_ligs):
    found, ini_state = find_state(olig, "initial")
    if not found: print("not found initial state for", idx, olig.formula)
    found, opt_state = find_state(olig, "b3lyp_opt")
    if not found: print("not found opt state for", idx, olig.formula)
    if not ini_state.natoms == opt_state.natoms or not opt_state.natoms == olig.natoms: 
        print(idx, olig.natoms, ini_state.natoms, opt_state.natoms)


2 26 26 28
15 45 45 34
19 41 41 31
21 11 11 49
27 19 19 40
34 26 26 41
39 40 40 25
101 38 38 41
109 27 27 55
112 38 38 26
124 49 49 11
131 20 20 30
152 31 31 78
176 29 29 49
237 42 42 92
248 14 14 39
266 68 68 28
284 25 25 26
298 36 36 19
299 31 31 20
316 13 13 50
362 40 40 29
412 34 34 12
437 35 35 25
463 50 50 35
516 46 46 35
538 34 34 42
548 27 27 80


In [20]:
ini_state.formula

'H6-C6-N2'

In [ ]:
print(ini_state.labels, ini_state.coord)


['N', 'N', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'H', 'H', 'H', 'H', 'H'] [[4.5006832, 6.5360133, 6.0969495], [4.8178039, 7.7631503, 8.199681], [5.5090233, 6.0874042, 6.7865811], [6.3546502, 4.9588405, 6.4981285], [7.3647485, 4.687261, 7.3343232], [7.6572895, 5.5100463, 8.4864511], [6.8827083, 6.5611596, 8.8334168], [5.7371124, 6.859897, 8.0142339], [4.3240109, 6.1658605, 5.4362939], [4.9739528, 8.288204, 8.9321226], [6.1527983, 4.465974, 5.7709214], [7.9245332, 3.963049, 7.1505586], [8.4373887, 5.290771, 9.0106385], [7.0526732, 7.1113595, 9.5415558]]


In [13]:
print(opt_state.labels, opt_state.coord)

['N', 'N', 'N', 'N', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'C', 'C', 'C', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'H', 'H'] [[4.138108, 19.446474, 6.560317], [1.732994, 19.611954, 7.880583], [0.443101, 23.137721, 4.663788], [2.395447, 23.020839, 3.587502], [3.092955, 23.207517, 2.875633], [5.304096, 19.351068, 5.942312], [6.012419, 18.61041, 6.339485], [5.665721, 20.137543, 4.828065], [6.64601, 20.015208, 4.358337], [4.750073, 21.060752, 4.351732], [4.993114, 21.691169, 3.489634], [3.49784, 21.181604, 4.991379], [3.229031, 20.335685, 6.118896], [2.466378, 22.090428, 4.601044], [1.23681, 22.187036, 5.258825], [0.944748, 21.356024, 6.392848], [1.93884, 20.425575, 6.82595], [0.580248, 19.683251, 8.525963], [0.453079, 19.002515, 9.379691], [1.15408, 23.634657, 3.661681], [0.059678, 26.519906, 0.660657], [1.284959, 26.658767, 1.323738], [1.633181, 25.729534, 2.377627], [0.744661, 

In [18]:
ini_state.get_moleclist()[0].view()

In [19]:
opt_state.moleclist[0].view()

In [11]:
def parse_ligand_optimization(new_calcs_folder, idx, ulig):
    log_path = None
    for r in range(10):
        tmp_path = str(new_calcs_folder+"Ulig_"+str(idx)+"_opt_r"+str(r)+"_LS.log")    
        if os.path.exists(tmp_path): log_path = tmp_path
    
    if log_path is None: 
        print(idx, "ulig log_path is None")
        return False, None
    else: 
        ## Parse Output
        lines = read_lines_file(log_path)
        output = g16_output(lines)
        labels, coord = output.get_last_geometry()
        energy = output.get_last_energy()
        
        ## Feed State Data
        exists, opt_state = find_state(ulig,"optimized")
        if exists:
            opt_state.set_geometry(labels, coord)
            opt_state.set_energy(energy,"hartree")
        else:
            opt_state = state(ulig, "optimized")
            opt_state.set_geometry(labels, coord)
            opt_state.set_energy(energy,"hartree")

    return exists, opt_state

In [12]:
def get_2nd_coord_sphere_idx(ulig): 

    #print(ulig.get_connected_idx())
    adjmat, adjnum = ulig.parent.get_adjmatrix()
    madjmat, madjnum = ulig.parent.get_metal_adjmatrix()
    indices = ulig.parent_indices
    for idx, at in enumerate(ulig.atoms):
        at.set_adjacencies(adjmat[ulig.parent_indices[idx]],madjmat[ulig.parent_indices[idx]],adjnum[ulig.parent_indices[idx]],madjnum[ulig.parent_indices[idx]])

    reg_adj = []
    for idx, at in enumerate(ulig.atoms):
        if idx in ulig.get_connected_idx():
            #print("ligand atom:", idx, at.label, at.adjacency, at.metal_adjacency)
            for a in at.adjacency:
            #    print(a, ulig.parent.labels[a])
                if a not in at.metal_adjacency and a not in reg_adj:
            #        print("added", idx)
                    reg_adj.append(a)  ## Indices of the molecule

    lig_indices = []
    for jdx in reg_adj:
        for kdx, pi in enumerate(ulig.parent_indices):
            if jdx == pi: lig_indices.append(kdx) 

    return lig_indices


In [13]:
def recover_ligand(ulig, debug: int=0):
    if debug > 0: print("Received ligand", ulig.formula)
    nml = deepcopy(ulig.parent.parent.get_moleclist())
    if debug > 0: print(len(nml),"molecules in moleclist")
    found = False
    for ml in nml:
        if debug > 0: print("A", ulig.parent.labels)
        if debug > 0: print("B", ml.labels)
        if ml.formula == ulig.parent.formula and all(x == y for x, y in zip(ulig.parent.labels, ml.labels)) and not found:
            if debug > 0: print("passed")
            nligs = deepcopy(ml.split_complex()[0])
            if debug > 0: print(ml.formula,"is candidate with", len(nligs), "ligands")
            for nl in nligs:
                if debug > 0: print("C", ulig.labels)
                if debug > 0: print("D", nl.labels)
                if nl.formula == ulig.formula and all(x == y for x, y in zip(ulig.labels, nl.labels)) and not found:
                    if debug > 0: print("ligand passed")
                    found = True
                    actual_ligand = deepcopy(nl)
                    return actual_ligand
                    if debug > 0: print(nl.formula,"is the same ligand")
    return None

### RMSD between initial and optimized distance matrices

In [14]:
for idx, ulig in enumerate(unique_ligands):
    if idx == 425: exists, opt_state = parse_ligand_optimization(new_calcs_folder, idx, ulig)

In [15]:
f1 = "/Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/"
p1 = "/Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/initial.xyz"
p2 = "/Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/optimized.xyz"

failed = []
for idx, ulig in enumerate(unique_ligands[0:10]):

    print("DOING", idx)
    if idx == 425: exists, opt_state = parse_ligand_optimization(new_calcs_folder, idx, ulig)

    if not hasattr(ulig,"parent_indices"):
        actual_ligand = recover_ligand(ulig, debug=1)
        if actual_ligand is None: break
        else: print("recovered")
    else:
        actual_ligand = deepcopy(ulig)                        
        print("deepcopied")
                
    idx_1    = actual_ligand.get_connected_idx()
    idx_2    = get_2nd_coord_sphere_idx(actual_ligand)
    joined_list_1 = idx_1 + idx_2

    labs_1   = extract_from_list(joined_list_1, actual_ligand.labels, dimension=1)
    coords_1 = extract_from_list(joined_list_1, actual_ligand.coord, dimension=1)

    exists, opt_state = find_state(actual_ligand,"optimized")
    if not exists:
        exists, opt_state = parse_ligand_optimization(new_calcs_folder, idx, actual_ligand)
    print(opt_state.labels)

    idx_1    = actual_ligand.get_connected_idx()
    idx_2    = get_2nd_coord_sphere_idx(actual_ligand)
    joined_list_2 = idx_1 + idx_2

    labs_2   = extract_from_list(joined_list_2, opt_state.labels, dimension=1)
    coords_2 = extract_from_list(joined_list_2, opt_state.coord, dimension=1)

    print(labs_1, labs_2)
    if all(x == y for x, y in zip(labs_1, labs_2)) and os.path.exists(p1) and os.path.exists(p2):
        writexyz(f1,"initial.xyz",labs_1, coords_1,0,1)
        writexyz(f1,"optimized.xyz",labs_2, coords_2, 0,1)
        command = " /Users/sergivela/Documents/Programes/Python/Python_Library/RMSD_Simple/calculate_rmsd.py /Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/initial.xyz /Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/optimized.xyz"
        #rmsd = subprocess.run("python"+command, capture_output=True, text=True)
        #rmsd = Popen(command, stdout=PIPE, shell=True, stderr=STDOUT, bufsize=1, close_fds=True)

        #print(rmsd)
        #%run -i '/Users/sergivela/Documents/Programes/Python/Python_Library/RMSD_Simple/calculate_rmsd.py' /Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/initial.xyz /Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/optimized.xyz --reorder
    else: 
        failed.append(idx)
        print("no", idx)
        #print(actual_ligand.labels)
        #print(opt_state.labels)

DOING 0
deepcopied
DOING 1
deepcopied
DOING 2
Received ligand H11-C11-N3-O
6 molecules in moleclist
A ['Fe', 'N', 'N', 'N', 'N', 'N', 'H', 'C', 'C', 'H', 'C', 'C', 'C', 'C', 'H', 'C', 'H', 'O', 'C', 'H', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'H', 'O', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'H', 'H', 'N', 'N', 'H', 'C', 'C', 'H', 'C', 'H', 'O', 'C', 'H', 'C', 'H', 'C', 'C', 'C', 'H', 'H', 'H', 'N', 'N', 'H', 'C', 'H', 'C', 'H', 'H', 'H', 'H']
B ['Cl', 'O', 'O', 'O', 'O']
A ['Fe', 'N', 'N', 'N', 'N', 'N', 'H', 'C', 'C', 'H', 'C', 'C', 'C', 'C', 'H', 'C', 'H', 'O', 'C', 'H', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'H', 'O', 'C', 'C', 'C', 'H', 'C', 'H', 'C', 'H', 'H', 'H', 'N', 'N', 'H', 'C', 'C', 'H', 'C', 'H', 'O', 'C', 'H', 'C', 'H', 'C', 'C', 'C', 'H', 'H', 'H', 'N', 'N', 'H', 'C', 'H', 'C', 'H', 'H', 'H', 'H']
B ['Cl', 'O', 'O', 'O', 'O']
A ['Fe', 'N', 'N', 'N', 'N', 'N', 'H', 'C', 

In [16]:
print(failed)

[2, 8]


In [13]:
#for idx, ulig in enumerate(unique_ligands[0:3]):
#
#    f1 = "/Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/"
#    p1 = "/Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/initial.xyz"
#    p2 = "/Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/optimized.xyz"
#
#    print("DOING", idx)
#
#    exists, opt_state = parse_ligand_optimization(new_calcs_folder, idx, ulig)
#    print(opt_state.labels)
#    actual_ligand = recover_ligand(ulig, opt_state, debug=0)
#    print(actual_ligand.labels)
#                
#    idx_1    = actual_ligand.get_connected_idx()
#    idx_2    = get_2nd_coord_sphere_idx(actual_ligand)
#    joined_list = idx_1 + idx_2
#    print(joined_list, actual_ligand.coord)
#
#    labs_1   = extract_from_list(joined_list, actual_ligand.labels, dimension=1)
#    coords_1 = extract_from_list(joined_list, actual_ligand.coord, dimension=1)
#    writexyz(f1,"initial.xyz",labs_1, coords_1,0,1)
#
#    print(joined_list, opt_state.coord)
#
#    labs_2   = extract_from_list(joined_list, opt_state.labels, dimension=1)
#    coords_2 = extract_from_list(joined_list, opt_state.coord, dimension=1)
#    writexyz(f1,"optimized.xyz",labs_2, coords_2, 0,1)
#
#    print(labs_1, coords_1)
#    print(labs_2, coords_2)
#    
#    if all(x == y for x, y in zip(labs_1, labs_2)) and os.path.exists(p1) and os.path.exists(p2):
#        command = " /Users/sergivela/Documents/Programes/Python/Python_Library/RMSD_Simple/calculate_rmsd.py /Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/initial.xyz /Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/optimized.xyz"
#        #rmsd = subprocess.run("python"+command, capture_output=True, text=True)
#        #rmsd = Popen(command, stdout=PIPE, shell=True, stderr=STDOUT, bufsize=1, close_fds=True)
#
#        #print(rmsd)
#        %run -i '/Users/sergivela/Documents/Programes/Python/Python_Library/RMSD_Simple/calculate_rmsd.py' /Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/initial.xyz /Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/optimized.xyz --reorder
#    else: 
#        print("no")
#        print(actual_ligand.labels)
#        print(opt_state.labels)

DOING 0
['S', 'N', 'C']
['S', 'N', 'C']
[1, 2] [[3.3374406, 11.5024855, 1.4276197], [3.9704876, 9.2347848, 2.9432491], [3.7377947, 10.164976, 2.3187184]]
[1, 2] [[8.361425, 4.288006, 15.115355], [5.722806, 4.295656, 14.040655], [6.819342, 4.293531, 14.488639]]
['N', 'C'] [[3.9704876, 9.2347848, 2.9432491], [3.7377947, 10.164976, 2.3187184]]
['N', 'C'] [[5.722806, 4.295656, 14.040655], [6.819342, 4.293531, 14.488639]]
0.02010563413991987
DOING 1
['N', 'N', 'N', 'N', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'C', 'H', 'H', 'C', 'H', 'C', 'H', 'C', 'C', 'H', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'C', 'H', 'H', 'C', 'H', 'C', 'H']
['N', 'N', 'N', 'N', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'C', 'H', 'H', 'C', 'H', 'C', 'H', 'C', 'C', 'H', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'H', 'C', 'C', 'H', 'H', 'C', 'H', 'C', 'H']
[0, 1, 2, 3, 16, 20, 4, 12, 24, 32, 13, 21, 33] [[2.2691288, 7.5949632, 4.5375795], [3.8675833, 6.2596114, 2.8450844], [4.6420522, 8.7876697, 5.75617

AttributeError: 'NoneType' object has no attribute 'labels'

In [21]:
ml = unique_ligands[2].parent.parent.get_moleclist()

In [13]:
mol = ml[4]
ligs = mol.split_complex()[0]
lig = ligs[0][0]
print(lig.formula)

H11-C11-N3-O


In [9]:
unique_ligands[2].parent

------------- Cell2mol MOLECULE Object --------------
 Version               = 0.1
 Type                  = specie
 Sub-Type               = molecule
 Number of Atoms       = 79
 Formula               = H33-C33-N9-O3-Fe
 Has Adjacency Matrix  = YES
 Has Parent            = YES
 Total Charge          = 2
 SMILES                = ['[H]C(=Nc1c([H])c([H])c(OC([H])([H])[H])c([H])c1[H])c1nc([H])c([H])n1[H]', '[H]C(=Nc1c([H])c([H])c(OC([H])([H])[H])c([H])c1[H])c1nc([H])c([H])n1[H]', '[H]C(=Nc1c([H])c([H])c(OC([H])([H])[H])c([H])c1[H])c1nc([H])c([H])n1[H]']
 # Ligands             = 3
 # Metals              = 1
---------------------------------------------------

In [10]:
for idx, ulig in enumerate(unique_ligands[0:9]):
    print(ulig.natoms, ulig.formula)

3 C-N-S
40 H18-C18-N4
26 H11-C11-N3-O
55 H24-C23-N8
61 H28-C25-N8
44 H24-C18-N2
60 H28-C26-N6
84 H44-C34-N6
31 H13-C13-N5


In [11]:
#from subprocess import Popen, PIPE, STDOUT
import io

# Create a buffer to capture stdout
output_buffer = io.StringIO()

# Temporarily redirect stdout to the buffer
sys.stdout = output_buffer

try: 
    for idx, ulig in enumerate(unique_ligands[0:9]):
        f1 = "/Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/"
        writexyz(f1,"initial.xyz",ulig.labels, ulig.coord,ulig.totcharge,1)
        p1 = "/Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/initial.xyz"
    
        found, state = find_state(ulig,"optimized")
        writexyz(f1,"optimized.xyz",ulig.labels, ulig.coord,ulig.totcharge,1)
        p2 = "/Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/optimized.xyz"
    
        if os.path.exists(p1) and os.path.exists(p2):
            command = " /Users/sergivela/Documents/Programes/Python/Python_Library/RMSD_Simple/calculate_rmsd.py /Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/initial.xyz /Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/optimized.xyz"
            #rmsd = subprocess.run("python"+command, capture_output=True, text=True)
            #rmsd = Popen(command, stdout=PIPE, shell=True, stderr=STDOUT, bufsize=1, close_fds=True)
    
            #print(rmsd)
            %run -i '/Users/sergivela/Documents/Programes/Python/Python_Library/RMSD_Simple/calculate_rmsd.py' /Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/initial.xyz /Users/sergivela/Documents/SCOPE/Database_SCO/test_reorder/optimized.xyz
        else: 
            print("no")

finally:
    # Reset stdout to its original value
    sys.stdout = sys.__stdout__

# Retrieve the captured output from the buffer
output = output_buffer.getvalue()

print(output)

3.698660438238847e-15
1.2630948273773117e-15
1.2403551861564225e-15
2.2240973997738707e-15
1.1343275741648675e-15
7.616881824556653e-16
1.7359553861149762e-15
1.547233415905513e-15
1.666985436533146e-15

3.698660438238847e-15
1.2630948273773117e-15
1.2403551861564225e-15
2.2240973997738707e-15
1.1343275741648675e-15
7.616881824556653e-16
1.7359553861149762e-15
1.547233415905513e-15
1.666985436533146e-15


In [19]:
for idx, ulig in enumerate(unique_ligands[0:10]):
    print(ulig.natoms)

In [20]:
output.split("\n")

['3.698660438238847e-15',
 '1.2630948273773117e-15',
 '1.2403551861564225e-15',
 '2.2240973997738707e-15',
 '1.1343275741648675e-15',
 '7.616881824556653e-16',
 '1.7359553861149762e-15',
 '1.547233415905513e-15',
 '1.666985436533146e-15',
 '6.949492130914263e-16',
 '']

In [20]:
# The current stdout
print(sys.stdout)

# The original stdout, before any redirection
print(sys.__stdout__)

In [19]:
res = output.split("\n")
type(res)
print(res)

AttributeError: 'list' object has no attribute 'getvalue'

In [18]:
import sys
sys.path.append("/Users/sergivela/Documents/Programes/Python/Python_Library/RMSD_Simple")

from calculate_rmsd import *

In [ ]:
main()